# UI Read Views

In [ ]:
dbutils.widgets.text("catalog", "access_drift")
catalog = dbutils.widgets.get("catalog")

context_schema = "context"
app_schema = "app"

## Create Schemas

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{context_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{app_schema}")

## List View: risk_case_list_view

In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.{app_schema}.risk_case_list_view AS
WITH gold AS (
  SELECT
    risk_case_key,
    drift_type,
    COALESCE(
      finding_type_label,
      CASE
        WHEN drift_type = 'nhi_residual_access' THEN 'NHI Access Drift'
        ELSE drift_type
      END
    ) AS drift_type_label,
    principal_kind,
    source_system,
    principal_type,
    risk_factors,
    subject_display_name,
    subject_type,
    subject_source_principal_id,
    owner_display_name,
    asset_source_id,
    asset_display_name,
    severity,
    detected_at
  FROM {catalog}.gold.gold_core
)
SELECT
  g.risk_case_key,
  CONCAT(g.drift_type_label, ' - ', g.subject_display_name, ' -> ', g.asset_display_name) AS card_title,
  g.drift_type_label,
  g.principal_kind,
  g.source_system,
  g.principal_type,
  g.risk_factors,
  g.subject_display_name AS identity_name,
  g.subject_type AS identity_type,
  g.severity,
  g.owner_display_name AS owner_name,
  g.detected_at,
  COALESCE(s.status, 'open') AS action_status,
  s.reviewer_name,
  s.reviewer_team,
  r.assignee_name,
  r.assignee_team,
  CASE g.severity
    WHEN 'CRITICAL' THEN 1
    WHEN 'HIGH' THEN 2
    WHEN 'MEDIUM' THEN 3
    WHEN 'LOW' THEN 4
    ELSE 99
  END AS severity_sort_order,
  g.severity IN ('CRITICAL', 'HIGH', 'MEDIUM') AS is_priority_visible
FROM gold g
LEFT JOIN {catalog}.{app_schema}.risk_action_state s
  ON g.risk_case_key = s.risk_case_key
LEFT JOIN {catalog}.{context_schema}.assignment_routing r
  ON r.subject_type = g.subject_type
 AND r.subject_id = g.subject_source_principal_id
 AND r.asset_id = g.asset_source_id
 AND r.drift_type = g.drift_type
""")

## Severity Distribution View: risk_severity_distribution_view

In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.{app_schema}.risk_severity_distribution_view AS
SELECT
  severity,
  COUNT(*) AS issue_count,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct,
  CASE severity
    WHEN 'CRITICAL' THEN 1
    WHEN 'HIGH' THEN 2
    WHEN 'MEDIUM' THEN 3
    WHEN 'LOW' THEN 4
    ELSE 99
  END AS severity_sort_order
FROM {catalog}.gold.gold_core
GROUP BY severity
""")

## Drift Distribution View: risk_drift_distribution_view

In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.{app_schema}.risk_drift_distribution_view AS
WITH gold AS (
  SELECT
    COALESCE(
      finding_type_label,
      CASE
        WHEN drift_type = 'nhi_residual_access' THEN 'NHI Access Drift'
        ELSE drift_type
      END
    ) AS drift_type_label,
    principal_kind,
    source_system,
    principal_type
  FROM {catalog}.gold.gold_core
)
SELECT
  drift_type_label,
  principal_kind,
  source_system,
  principal_type,
  COUNT(*) AS drift_count
FROM gold
GROUP BY drift_type_label, principal_kind, source_system, principal_type
""")

## Access Path View: risk_access_path_view

In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.{app_schema}.risk_access_path_view AS
WITH gold AS (
  SELECT
    *,
    COALESCE(
      finding_type_label,
      CASE
        WHEN drift_type = 'nhi_residual_access' THEN 'NHI Access Drift'
        ELSE drift_type
      END
    ) AS drift_type_label
  FROM {catalog}.gold.gold_core
)
SELECT
  g.risk_case_key,
  g.severity,
  g.drift_type,
  g.drift_type_label,
  g.principal_kind,
  g.source_system,
  g.principal_type,
  g.risk_factors,
  g.owner_principal_id,
  g.owner_display_name,
  g.subject_principal_id,
  g.subject_source_principal_id,
  g.subject_display_name,
  g.subject_type,
  subj.principal_type AS subject_object_type,
  subj.status AS subject_status,
  g.credential_id,
  cred.credential_type,
  cred.status AS credential_status,
  cred.is_active AS credential_is_active,
  cred.expires_at AS credential_expires_at,
  g.asset_id,
  g.asset_source_id,
  g.asset_display_name,
  g.asset_environment,
  g.asset_sensitivity,
  owner_edge.edge_id AS owner_edge_id,
  owner_edge.relationship AS owner_edge_relationship,
  owner_edge.status AS owner_edge_status,
  owner_edge.evidence_source AS owner_edge_evidence_source,
  access_edge.edge_id AS access_edge_id,
  access_edge.relationship AS access_edge_relationship,
  access_edge.permission AS access_permission,
  access_edge.status AS access_edge_status,
  access_edge.evidence_source AS access_edge_evidence_source,
  g.detected_at
FROM gold g
LEFT JOIN {catalog}.silver.silver_edges owner_edge
  ON g.owner_edge_id = owner_edge.edge_id
LEFT JOIN {catalog}.silver.silver_edges access_edge
  ON g.access_edge_id = access_edge.edge_id
LEFT JOIN {catalog}.silver.silver_credentials cred
  ON g.credential_id = cred.credential_id
LEFT JOIN {catalog}.silver.silver_principals subj
  ON g.subject_principal_id = subj.principal_id
""")

## Detail View: risk_card_detail_view

In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.{app_schema}.risk_card_detail_view AS
WITH gold AS (
  SELECT
    *,
    COALESCE(
      finding_type_label,
      CASE
        WHEN drift_type = 'nhi_residual_access' THEN 'NHI Access Drift'
        ELSE drift_type
      END
    ) AS drift_type_label
  FROM {catalog}.gold.gold_core
)
SELECT
  g.risk_case_key,
  g.finding_id,
  g.severity,
  g.drift_type,
  g.drift_type_label,
  g.principal_kind,
  g.source_system,
  g.principal_type,
  g.risk_factors,
  g.subject_display_name,
  g.subject_type,
  subj.principal_type AS subject_object_type,
  subj.status AS nhi_status,
  g.owner_display_name,
  own.status AS owner_account_status,
  own.termination_date AS owner_termination_date,
  g.asset_display_name,
  ast.asset_type,
  ast.environment AS asset_environment,
  ast.sensitivity AS asset_sensitivity,
  cred.credential_type,
  cred.is_active AS credential_is_active,
  acc.permission AS access_permission,
  acc.status AS access_status,
  f.owner_status AS factor_owner_status,
  f.credential_is_active AS factor_credential_active,
  f.permission AS factor_permission,
  f.asset_sensitivity AS factor_asset_sensitivity,
  UPPER(g.asset_sensitivity) IN ('CRITICAL', 'HIGH') AS factor_sensitive_access,
  ctx.data_scope_label,
  ctx.sensitive_data_examples,
  ctx.sensitivity_reason,
  g.evidence_summary AS risk_summary,
  g.recommended_action AS deterministic_recommended_action,
  COALESCE(s.status, 'open') AS action_status,
  s.reviewer_name,
  s.reviewer_team,
  s.updated_at AS action_updated_at,
  r.assignee_name,
  r.assignee_team,
  r.routing_reason,
  g.detected_at
FROM gold g
LEFT JOIN {catalog}.intermediate.nhi_residual_access_findings f
  ON g.finding_id = f.finding_id
LEFT JOIN {catalog}.silver.silver_principals subj
  ON g.subject_principal_id = subj.principal_id
LEFT JOIN {catalog}.silver.silver_principals own
  ON g.owner_principal_id = own.principal_id
LEFT JOIN {catalog}.silver.silver_credentials cred
  ON g.credential_id = cred.credential_id
LEFT JOIN {catalog}.silver.silver_assets ast
  ON g.asset_id = ast.asset_id
LEFT JOIN {catalog}.silver.silver_edges acc
  ON g.access_edge_id = acc.edge_id
LEFT JOIN {catalog}.{context_schema}.asset_context ctx
  ON ctx.source_asset_id = g.asset_source_id
LEFT JOIN {catalog}.{app_schema}.risk_action_state s
  ON g.risk_case_key = s.risk_case_key
LEFT JOIN {catalog}.{context_schema}.assignment_routing r
  ON r.subject_type = g.subject_type
 AND r.subject_id = g.subject_source_principal_id
 AND r.asset_id = g.asset_source_id
 AND r.drift_type = g.drift_type
""")

## Smoke Checks

In [ ]:
display(spark.sql(f"SELECT COUNT(*) AS row_count FROM {catalog}.{app_schema}.risk_case_list_view"))
display(spark.sql(f"SELECT * FROM {catalog}.{app_schema}.risk_severity_distribution_view ORDER BY severity_sort_order"))
display(spark.sql(f"SELECT * FROM {catalog}.{app_schema}.risk_drift_distribution_view ORDER BY drift_count DESC, drift_type_label"))
display(spark.sql(f"SELECT COUNT(*) AS row_count FROM {catalog}.{app_schema}.risk_access_path_view"))
display(spark.sql(f"SELECT COUNT(*) AS row_count FROM {catalog}.{app_schema}.risk_card_detail_view"))